In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

from pathlib import Path
import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2
Configuration: {'general': {'run_name': 'experiment_with_03_classes', 'seed': 42, 'n_classes': 3}, 'dataset': {'split_type': 'test', 'test_split': 0.2, 'cutoff_year': 1996}, 'paths': {'data_exploration_dir': 'output/experiment_with_03_classes/data_exploration', 'artifacts_dir': 'output/experiment_with_03_classes/artifacts', 'embeddings_dir': 'output/experiment_with_03_classes/embeddings', 'models_dir': 'output/experiment_with_03_classes/models', 'results_dir': 'output/experiment_with_03_classes/results'}, 'model': {'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False}, 'training': {'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3'}, 'evaluation': {'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}}

=== Configuration Variables ===

[DATASET]
  DATASET_

# 06 – Semantic Search Demo

Interactive semantic search on the Reuters documents using a MiniLM embedding model. Optionally, this notebook can switch to a Retrieval‑Augmented Generation (RAG) mode that synthesises an answer from the top retrieved documents.

In [2]:

import json, pathlib, faiss
from sentence_transformers import SentenceTransformer
from src.datasets.dataset import get_dataset

from src.rag import _SBERT_DIR, _EMBEDDINGS_DIR

# Use the correct embeddings directory from the configuration
INDEX_PATH = _SBERT_DIR / 'index.faiss'
DOCS_PATH = _SBERT_DIR / 'meta.jsonl'
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'


INFO | Loading faiss with AVX2 support.
INFO | Successfully loaded faiss with AVX2 support.
INFO | Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.
/home/marcmaceira/projects/reuters-rag-classifier_clean_v2/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load documents (train + test for demo)
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES,
    cutoff_year=GENERAL_CUTOFF_YEAR
)

INFO | Loading Reuters dataset with configuration:
INFO |   - Split type: test
INFO |   - Number of classes: 3
INFO |   - Samples per class: 20
INFO |   - Random seed: None
INFO | Loading small test dataset with 20 samples per class across 3 classes
INFO | Selected classes: earn, acq, crude
INFO |   - Class 'earn': 14 train, 6 test
INFO |   - Class 'acq': 14 train, 6 test
INFO |   - Class 'crude': 14 train, 6 test


In [4]:
# Combine train and test documents
docs = X_train + X_test
labels = y_train + y_test

if not INDEX_PATH.exists():
    print('Building embeddings…')
    model = SentenceTransformer(MODEL_NAME)
    emb = model.encode(docs, show_progress_bar=True, batch_size=64, convert_to_numpy=True)
    dimension = emb.shape[1]
    index = faiss.IndexFlatIP(dimension)
    # normalise for cosine sim
    faiss.normalize_L2(emb)
    index.add(emb)
    
    # Create metadata for each document
    meta = []
    for i, (txt, label) in enumerate(zip(docs, labels)):
        meta.append({
            "id": i,
            "text": txt,
            "label": label,
            "vector": emb[i].tolist()  # Store the vector in the metadata
        })
    
    # Ensure directory exists
    INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
    
    # Save FAISS index
    faiss.write_index(index, str(INDEX_PATH))
    
    # Save metadata as JSONL
    with open(DOCS_PATH, 'w') as f:
        for m in meta:
            f.write(json.dumps(m) + '\n')
            
    print(f'Index and docs saved to {_SBERT_DIR}')
else:
    print('Embeddings already built. Loading…')
    index = faiss.read_index(str(INDEX_PATH))
    # Load metadata
    meta = []
    with open(DOCS_PATH) as f:
        for line in f:
            meta.append(json.loads(line))
    docs = [m['text'] for m in meta]
    model = SentenceTransformer(MODEL_NAME)


Embeddings already built. Loading…


NameError: name 'json' is not defined

In [5]:
# Function to search for similar documents
def search(query: str, k: int = 5):
    # Encode query
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    
    # Search
    D, I = index.search(q_emb, k)
    
    # Return results
    results = []
    for score, idx in zip(D[0], I[0]):
        results.append({
            'text': meta[idx]['text'],
            'label': meta[idx]['label'],
            'score': float(score)
        })
    return results

In [6]:

# Demo
search("oil prices in saudi arabia")


Batches: 100%|██████████| 1/1 [00:00<00:00, 69.65it/s]

Top 5 results:
1. (score=0.486) RECENT U.S. OIL DEMAND UP 0.1 PCT FROM YEAR AGO
  U.S. oil demand as measured by
  products supplied rose 0.1 pct in the four weeks ended March 20
  to 16.16 mln barrels per day from 16.15 mln in the …

2. (score=0.465) KUWAIT SAYS OPEC 2.4 MLN BPD BELOW CEILING
  Kuwaiti oil minister Sheikh Ali
  al-Khalifa al-Sabah said OPEC was producing well below its oil
  output ceiling and this would help prices move higher,
 …

3. (score=0.450) VENEZUELA SEES OIL STABILITY DESPITE GULF ATTACK
  Venezuelan Energy Minister Arturo
  Hernandez Grisanti said he foresaw market stability in the
  price of crude, despite growing tension in the Gulf …

4. (score=0.442) SUPPLIES, MIDEAST TENSION FUEL GAINS IN OIL
  Petroleum futures rallied today in a
  market that was expecting declines in domestic supplies and
  became further unsettled by escalated Mideast fightin…

5. (score=0.418) SOUTHLAND &lt;SLC> UNIT RAISES CRUDE PRICES
  Southland Corp's Citgo Petrleum Corp
  sai

### Optional – Retrieval‑Augmented Generation (RAG)
If you have an OpenAI key configured, you can uncomment the cell below to generate answers from the retrieved passages.

In [ ]:
# !pip install openai
from openai import OpenAI
import textwrap
import os

# Initialize the OpenAI client
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY env var not set.')

def rag_answer(question: str, k: int = 5) -> str:
    q_emb = model.encode([question], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, k)
    
    # Collect reference documents
    references = []
    for rank, (idx, score) in enumerate(zip(I[0], D[0]), 1):
        references.append(f"Document {rank} (score={score:.3f}): {docs[idx]}")
    
    # Join all reference documents for context
    context = "\n".join([docs[i] for i in I[0]])
    prompt = f"Answer the question based only on the context below.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    
    # Generate answer
    response = client.completions.create(
        model='gpt-3.5-turbo-instruct',
        prompt=prompt,
        max_tokens=256
    )
    answer = textwrap.dedent(response.choices[0].text).strip()
    
    # Return both the answer and the reference documents
    full_response = f"Answer:\n{answer}\n\nReference Documents:\n" + "\n\n".join(references)
    return full_response

# Example - uncomment to test
print(rag_answer("What is the effects of the Regan administration?"))

Batches: 100%|██████████| 1/1 [00:00<00:00, 71.83it/s]
INFO | HTTP Request: POST https://api.openai.com/v1/completions "HTTP/1.1 200 OK"


Answer:
The Regan administration implemented policies that aimed to increase domestic oil production and reduce the country's dependence on foreign oil. They also explored tax incentives for the oil and gas industry and advocated for a higher fill rate of the Strategic Petroleum Reserve. The administration also faced diplomatic tensions and potential military confrontation in the Middle East, particularly with Iran. Additionally, the administration was involved in the proposed takeover of Taft Broadcasting Co and saw a stable market for Dutch food retailer Ahold NV despite fluctuations in the dollar value.

Reference Documents:
Document 1 (score=0.211): HERRINGTON SAYS HE MAY CALL FOR OIL TAX BENEFITS
  Energy Secretary John Herrington
  said he may recommend to the White House that the domestic oil
  industry be given tax benefits to help it produce more oil and
  head off increasing U.S. dependence on foreign oil.
      He said also at a news conference that he would recommend
  to t